# Kaggle: `sft_lora_fp` -- full-precision LoRA (no 4-bit) -> evaluate

Ablation run: same SFT training as `sft_qlora`, but the base model loads in full **bf16 precision** instead of 4-bit NF4 quantization (`peft.lora_fullprecision_r8` in `config.yaml`'s `runs:` list -- `--load-in-4bit` simply omitted from `train_sft.py`'s args). Tests whether QLoRA's 4-bit quantization is costing any quality versus full-precision LoRA, isolated from every other variable (same data, same LoRA rank/alpha/dropout, same hyperparameters).

Baseline and `sft_qlora` are already done (`results/summary.csv`) -- this notebook only adds the `sft_lora_fp` row.

**Elevated OOM risk versus every prior run**: the base model in bf16 is roughly 4x the memory of the 4-bit-quantized weights used everywhere else so far (~6GB vs. ~1.5GB for a 3B model). Nothing else has been run at this memory footprint on a T4 yet. The dry-run cell matters more than usual here -- if it OOMs, drop `per_device_batch_size` (currently 2, from `config.yaml`) before the full run, don't just retry blind.

**Before running:**
1. Zip `src/` (contents) + `config.yaml` as `src.zip` (same as every prior notebook).
2. Upload that zip plus `data/splits/sft_train.jsonl`, `data/splits/sft_valid.jsonl`, `data/splits/sft_test.jsonl` as a Kaggle Dataset.
3. Single T4 accelerator, Internet on.
4. Run all cells. Watch the dry run's timing *and* whether it OOMs before letting the full run proceed.

**Output:** `/kaggle/working/adapters/sft_lora_fp/` (download to `adapters/sft_lora_fp/` locally), `/kaggle/working/results/` (`sft_lora_fp_gen.jsonl`, `sft_lora_fp_scored.jsonl`, `summary.csv` with just the `sft_lora_fp` row -- merge into local `results/summary.csv`).

In [ ]:
!pip install -q -U "trl==1.10.0" "peft==0.20.0" "transformers==5.15.0" "bitsandbytes==0.50.1" "accelerate==1.14.0" pyyaml
# peft 0.20.0 requires torchao>=0.16.0 for one of its internal LoRA-dispatch
# checks, but Kaggle's base image ships torchao==0.10.0 -- that check only
# gets reached when loading a PEFT adapter onto a full-precision base model
# (this run's whole point), not a 4-bit-quantized one, so sft_qlora/
# dpo_from_sft never hit it. This project doesn't use torchao's own
# quantization scheme at all (only bitsandbytes NF4), so nothing depends on
# it -- uninstalling it here avoids the crash entirely rather than chasing
# a compatible version to upgrade to. Found the hard way after a full
# 3h15m training run completed successfully and then crashed only at eval
# generation. See LOG.md 2026-08-19.
!pip uninstall -y -q torchao
import torch
print("CUDA available:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Unsloth (accelerated QLoRA/LoRA path)

Same install + CUDA-survived check as every other notebook -- `--no-deps` prevents pip from touching the already-working `torch`. See `LOG.md` 2026-08-18/19 for why this exists and why it's non-negotiable (broke CUDA on two separate real sessions, once even with `--no-deps` for an unrelated Kaggle-side reason -- check the *first* cell's CUDA line too if this one fails, don't assume it's always the Unsloth install).

In [ ]:
!pip install -q --no-deps unsloth unsloth_zoo
import torch
print("CUDA available after unsloth install:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), (
    "CUDA broke after installing unsloth -- see LOG.md 2026-08-18/19. Do not proceed with "
    "--use-unsloth training if this assertion fails; fall back to USE_UNSLOTH=False instead, or "
    "check whether the *first* cell's CUDA line was already False (a different, Kaggle-side issue)."
)

In [ ]:
import os, sys, zipfile

def find_repo(root="/kaggle/input"):
    repo_dir = None
    config_path = None
    zip_path = None
    for r, dirs, files in os.walk(root):
        if "build_irac.py" in files and os.path.basename(r) == "data":
            src_dir = os.path.dirname(r)
            if os.path.basename(src_dir) == "src":
                repo_dir = os.path.dirname(src_dir)
        if config_path is None and "config.yaml" in files:
            config_path = os.path.join(r, "config.yaml")
        if "src.zip" in files:
            zip_path = os.path.join(r, "src.zip")
    return repo_dir, config_path, zip_path

repo_dir, config_path, zip_path = find_repo()

if repo_dir is None and zip_path:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("/kaggle/working/repo")
    repo_dir, config_path, _ = find_repo("/kaggle/working/repo")
    print("Extracted", zip_path)

if repo_dir is None or config_path is None:
    raise FileNotFoundError(
        f"repo_dir={repo_dir}, config_path={config_path} -- could not find both "
        "src/data/build_irac.py and config.yaml under /kaggle/input (in any "
        "nesting). Check the dataset is attached. If just attached/updated, "
        "try Restart & Run All."
    )

REPO_DIR = repo_dir
sys.path.insert(0, REPO_DIR)
print("REPO_DIR =", REPO_DIR)
print("config.yaml at", config_path)

import yaml
with open(config_path) as f:
    cfg = yaml.safe_load(f)

MODEL_ID = cfg["model"]["candidates"][cfg["model"]["active"]]["hf_id"]
print("Active model:", MODEL_ID)

In [ ]:
import os

def find_data_file(name, root="/kaggle/input"):
    for r, dirs, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    raise FileNotFoundError(f"{name} not found under {root} -- check it was included in the uploaded dataset.")

TRAIN_FILE = find_data_file("sft_train.jsonl")
VALID_FILE = find_data_file("sft_valid.jsonl")
TEST_FILE = find_data_file("sft_test.jsonl")
print(TRAIN_FILE, VALID_FILE, TEST_FILE, sep="\n")

RESULTS_DIR = "/kaggle/working/results"
ADAPTERS_DIR = "/kaggle/working/adapters"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(ADAPTERS_DIR, exist_ok=True)
SUMMARY_CSV = os.path.join(RESULTS_DIR, "summary.csv")

from src.eval.generate import run as generate_run
from src.eval.score import score_file, append_summary_row

GEN_BATCH_SIZE = 16

## SFT training (`sft_lora_fp`, full precision, no 4-bit)

Hyperparameters from `config.yaml`'s `peft`/`training.sft` sections, same as `sft_qlora` -- the *only* difference from that run is `--load-in-4bit` being omitted below. `--use-unsloth` stays on (Unsloth supports full 16-bit LoRA, not just 4-bit -- this isn't a QLoRA-only tool).

In [ ]:
peft_cfg = cfg["peft"]
sft_cfg = cfg["training"]["sft"]
max_seq_length = cfg["training"]["max_seq_length"]

SFT_LORA_FP_DIR = os.path.join(ADAPTERS_DIR, "sft_lora_fp")

USE_UNSLOTH = True

def train_sft_args(output_dir, max_steps=None, num_epochs=None):
    args = [
        "--model", MODEL_ID,
        "--train-file", TRAIN_FILE,
        "--eval-file", VALID_FILE,
        "--output-dir", output_dir,
        # NOTE: no --load-in-4bit here -- this is the entire point of this run.
        "--lora-r", str(peft_cfg["lora_r"]),
        "--lora-alpha", str(peft_cfg["lora_alpha"]),
        "--lora-dropout", str(peft_cfg["lora_dropout"]),
        "--target-modules", *peft_cfg["target_modules"],
        "--max-seq-length", str(max_seq_length),
        "--per-device-batch-size", str(sft_cfg["per_device_batch_size"]),
        "--gradient-accumulation-steps", str(sft_cfg["gradient_accumulation_steps"]),
        "--learning-rate", str(sft_cfg["learning_rate"]),
    ]
    if USE_UNSLOTH:
        args += ["--use-unsloth"]
    if max_steps is not None:
        args += ["--max-steps", str(max_steps)]
    else:
        args += ["--num-epochs", str(num_epochs)]
    return args

print(train_sft_args(SFT_LORA_FP_DIR, num_epochs=sft_cfg["epochs"]))

In [ ]:
import subprocess, sys

env = os.environ.copy()
env["PYTHONPATH"] = REPO_DIR
env["PYTHONUNBUFFERED"] = "1"

dryrun_dir = os.path.join(ADAPTERS_DIR, "sft_lora_fp_dryrun")
cmd = [sys.executable, "-m", "src.train.train_sft"] + train_sft_args(dryrun_dir, max_steps=5)
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=env)

In [ ]:
# Full training run. Check the dry-run cell's per-step timing above and the
# GPU-memory diagnostic lines (GPU 0 vs GPU 1) before running this. If the
# dry run OOM'd, drop --per-device-batch-size (currently read from
# config.yaml's training.sft.per_device_batch_size, 2) before retrying --
# don't just rerun the same command. See this notebook's intro markdown for
# why this run has more OOM risk than any prior one (full bf16 base model,
# not 4-bit quantized).
cmd = [sys.executable, "-m", "src.train.train_sft"] + train_sft_args(SFT_LORA_FP_DIR, num_epochs=sft_cfg["epochs"])
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=env)

## `sft_lora_fp` eval

Same test split, same generation script, `sft_lora_fp` adapter attached instead of `sft_qlora`.

In [ ]:
SFT_LORA_FP_GEN = os.path.join(RESULTS_DIR, "sft_lora_fp_gen.jsonl")

generate_run(
    input_path=TEST_FILE,
    output_path=SFT_LORA_FP_GEN,
    model_id=MODEL_ID,
    adapter_path=SFT_LORA_FP_DIR,
    load_in_4bit=False,
    batch_size=GEN_BATCH_SIZE,
    max_new_tokens=350,
    limit=None,
)

In [ ]:
SFT_LORA_FP_SCORED = os.path.join(RESULTS_DIR, "sft_lora_fp_scored.jsonl")
sft_lora_fp_summary = score_file(SFT_LORA_FP_GEN, SFT_LORA_FP_SCORED)
append_summary_row("sft_lora_fp", sft_lora_fp_summary, SUMMARY_CSV)
print("sft_lora_fp:", sft_lora_fp_summary)

In [ ]:
import pandas as pd
df = pd.read_csv(SUMMARY_CSV)
print(df.to_string(index=False))